# HSE HPC: Schweinhart Synthetic Controls

This is the HPC wrapper for the same `agent_language_geometry.schweinhart_controls` implementation validated locally. It runs sphere, Swiss roll, Sierpinski carpet, and Menger sponge controls at 100,000 points. It is an Appendix-A-style validation, not an exact numerical reproduction of Gromov et al.: their exact random samples and sample-size ladder are not published.

Required resources: CPU only; 10 cores and the 30-minute HSE `test` queue are sufficient for this validation run.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = 'https://github.com/shavrinaanastasia/LANC.git'
REPO = Path.home() / 'LANC'

if not REPO.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '--user', '-q', 'editables==0.5'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'], cwd=REPO, check=True)
print('Repository:', REPO)
print('Python:', sys.version)

In [ ]:
import numpy as np
import scipy

print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('CPU count visible to notebook:', __import__('os').cpu_count())

In [ ]:
from agent_language_geometry.schweinhart_controls import KNOWN_DIMENSIONS, run_schweinhart_controls

POINTS = 100_000
SAMPLE_SIZES = [10_000, 20_000, 40_000, 60_000, 80_000, 100_000]
ALPHAS = list(range(1, 11))
SEED = 20260926

controls = run_schweinhart_controls(POINTS, SAMPLE_SIZES, ALPHAS, SEED)
print('Completed deterministic synthetic-control calculation.')

In [ ]:
import pandas as pd

rows = []
for shape, result in controls.items():
    for estimate in result['alpha_estimates']:
        estimate_dimension = estimate['dimension']
        rows.append({
            'shape': shape,
            'known_dimension': result['known_dimension'],
            'alpha': estimate['alpha'],
            'estimated_dimension': estimate_dimension,
            'ape_percent': abs(estimate_dimension - result['known_dimension']) / result['known_dimension'] * 100,
            'r_squared': estimate['r_squared'],
            'ci_95': estimate['dimension_ci_95'],
        })

table = pd.DataFrame(rows)
table

In [ ]:
summary = table.groupby('shape', sort=False).agg(
    known_dimension=('known_dimension', 'first'),
    estimate_min=('estimated_dimension', 'min'),
    estimate_max=('estimated_dimension', 'max'),
    best_ape_percent=('ape_percent', 'min'),
    mean_ape_percent=('ape_percent', 'mean'),
    min_r_squared=('r_squared', 'min'),
)
summary

## Acceptance check

The implementation passes a basic numerical control when every estimate is finite, every 95% dimension interval is available, and every shape has a low-error region near its known dimension. Matching Gromov Table A1 exactly is not required or expected because their simulation ladder and random samples are not public.

In [ ]:
assert set(controls) == set(KNOWN_DIMENSIONS)
assert table['estimated_dimension'].notna().all()
assert table['ci_95'].notna().all()
assert (summary['best_ape_percent'] < 5).all()
print('PASS: every known shape has a Schweinhart estimate within 5% at at least one alpha.')